# Task 4 · Sentiment Analysis on Social-Media / Review Comments

**Goal:** build a machine-learning model that reads a short piece of text and decides whether the sentiment is **negative, neutral or positive**.

**What I'm doing in this notebook**

1. Load the data and look at the class balance
2. Clean the text (lowercase, punctuation, stopwords, tokenise, lemmatise)
3. Turn text into numbers with TF-IDF
4. Train Naive Bayes, Logistic Regression and a Linear SVM (80/20 split)
5. Compare them with accuracy / precision / recall / F1 and confusion matrices
6. Look at the mistakes the best model makes and try to understand why
7. Conclude and talk about real-world use

**Dataset:** `sentiment_data.csv` – about 241k comments with a `Sentiment` column coded as `0, 1, 2`.
Reading a few comments from each code makes the meaning clear, so I'm using:

| code | meaning  |
|------|----------|
| 0    | Negative |
| 1    | Neutral  |
| 2    | Positive |

*(For example "…get seriously annoyed store doesn't take apple pay" is coded 0, and "great video love style everything" is coded 2.)*

In [ ]:
# Only needed once, if something is missing in your environment:
# !pip install pandas numpy scikit-learn nltk wordcloud matplotlib seaborn textblob

In [ ]:
import re
import time
import random
import warnings
from collections import Counter
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42          # so my results are reproducible
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# one colour per class, reused in every chart so it is easy to read
CLASS_NAMES = ["Negative", "Neutral", "Positive"]
PALETTE = {"Negative": "#c0504d", "Neutral": "#7f8c8d", "Positive": "#4f9d69"}

## 1 · Load the data and check the class distribution

In [ ]:
# Look for the file next to the notebook first, then in the upload folder
candidates = [Path("sentiment_data.csv"), Path("/mnt/user-data/uploads/sentiment_data.csv")]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("sentiment_data.csv not found - put it in the same folder as this notebook.")

raw = pd.read_csv(data_path)
print("Raw shape:", raw.shape)
raw.head()

In [ ]:
# The first column is just a saved index ("Unnamed: 0"), so I drop it.
# I also give the columns shorter names and add a readable label.
label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

df = (
    raw.drop(columns=["Unnamed: 0"], errors="ignore")
       .rename(columns={"Comment": "text", "Sentiment": "label"})
)
df["sentiment"] = df["label"].map(label_map)

print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate comments:", df["text"].duplicated().sum())

In [ ]:
# Missing comments are useless for a text model, so they go.
df = df.dropna(subset=["text"]).reset_index(drop=True)
df["text"] = df["text"].astype(str)
print("Rows after dropping empty comments:", len(df))

In [ ]:
counts = df["sentiment"].value_counts().reindex(CLASS_NAMES)
percent = (counts / counts.sum() * 100).round(1)
print(pd.DataFrame({"count": counts, "percent": percent}))

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(CLASS_NAMES, counts.values, color=[PALETTE[c] for c in CLASS_NAMES])

for bar, c, p in zip(bars, counts.values, percent.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + counts.max() * 0.01,
            f"{c:,}\n({p}%)", ha="center", va="bottom", fontsize=10)

ax.set_ylim(0, counts.max() * 1.18)
ax.set_title("Sentiment distribution in the dataset")
ax.set_ylabel("Number of comments")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Quick look at comment length per class (number of words)
df["n_words"] = df["text"].str.split().str.len()
df.groupby("sentiment")["n_words"].agg(["mean", "median", "max"]).reindex(CLASS_NAMES).round(1)

**What I notice**

* The data is **not perfectly balanced**: positive is the biggest class (about 43%), then neutral (about 34%), and negative is the smallest (about 23%).
  A model can look good just by leaning towards "positive", so I will not trust accuracy alone – I'll look at **macro-averaged F1** (which gives each class equal weight) and per-class recall as well.
* Neutral comments are noticeably **shorter** on average than positive/negative ones. That makes sense: people who feel strongly tend to write more, while neutral comments are often just a statement or a headline.
* The comments come from a mix of topics (payments apps, politics, tech/ML videos). So the model will learn some topic words along with the real sentiment words – something to keep in mind later.

## 2 · Text preprocessing

Raw text is messy, and a model treats `"Great"`, `"great"` and `"great!!"` as three different things. The pipeline I'm using:

| step | why |
|------|-----|
| **lowercase** | `Good` and `good` should be the same word |
| **remove links, punctuation, digits** | they add noise, not sentiment |
| **tokenise** | split the sentence into separate words |
| **remove stopwords** | words like *the, is, and* carry almost no sentiment |
| **lemmatise** | `loved, loving, loves → love`, so the vocabulary is smaller |

Two small decisions I made on purpose:

* **I keep negation words** (`not`, `no`, `nor`). NLTK's default stopword list removes them, but "not good" and "good" are opposites, so throwing `not` away would hurt a sentiment model.
* I used **lemmatisation instead of stemming** because it returns real words (`studies → study`), which is easier to read in the word clouds and feature lists later. Stemming would be faster but gives things like `studi`.

In [ ]:
# Download the NLTK resources (quiet if they are already there)
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

# Default English stopwords, minus the negations I want to keep
stop_words = set(stopwords.words("english")) - {"no", "nor", "not"}

lemmatizer = WordNetLemmatizer()
tokenizer = RegexpTokenizer(r"[a-z]+")   # keeps only letters, so punctuation and digits disappear


@lru_cache(maxsize=None)
def lemma(word):
    # cache the result - the same words repeat thousands of times in 240k comments
    return lemmatizer.lemmatize(word)


def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)      # links
    text = text.replace("\u2019", "").replace("'", "")   # don't / don\u2019t -> dont (stays one token)
    tokens = tokenizer.tokenize(text)                    # letters only
    tokens = [lemma(t) for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(tokens)


# sanity check on a few made-up sentences before running it on everything
for sample in ["I DON'T like this phone!!! http://abc.com", "The movies were not good, 10/10 would avoid."]:
    print(sample, "->", clean_text(sample))

In [ ]:
start = time.time()
df["clean_text"] = df["text"].apply(clean_text)
print(f"Cleaned {len(df):,} comments in {time.time() - start:.1f} seconds")

# before / after for a few random rows
df[["text", "clean_text"]].sample(5, random_state=RANDOM_STATE)

### Tidying up after cleaning

Cleaning can create a few new problems, so I check for them before modelling:

* some comments might become **empty** (they only contained stopwords)
* the same comment can appear many times – if one copy lands in the training set and another in the test set, the model gets to "see the answer" and the score looks better than it really is (**data leakage**)
* a few identical comments carry **different labels**, which only confuses the model

In [ ]:
n_start = len(df)

# 1) empty after cleaning
df = df[df["clean_text"].str.len() > 0]
n_empty = n_start - len(df)

# 2) identical text but different labels -> remove all copies (we cannot know which label is right)
conflict = df.groupby("clean_text")["label"].transform("nunique") > 1
n_conflict = int(conflict.sum())
df = df[~conflict]

# 3) exact duplicates -> keep one copy
n_before_dedupe = len(df)
df = df.drop_duplicates(subset="clean_text").reset_index(drop=True)
n_dupes = n_before_dedupe - len(df)

print(f"Started with           : {n_start:,}")
print(f"Removed empty          : {n_empty:,}")
print(f"Removed conflicting    : {n_conflict:,}")
print(f"Removed duplicates     : {n_dupes:,}")
print(f"Rows used for modelling: {len(df):,}")
print()
print(df["sentiment"].value_counts().reindex(CLASS_NAMES))

### Word clouds for each sentiment

The biggest words are the most frequent ones. One problem: a few words (like the names of the main topics) are the most common in **all** classes and would make the three clouds look identical.
So, instead of guessing, I find the words that appear in the top-30 of *every* class and hide only those. What's left shows what is **special** to each sentiment.

In [ ]:
words_by_class = {
    s: Counter(w for t in df.loc[df["sentiment"] == s, "clean_text"] for w in t.split())
    for s in CLASS_NAMES
}

# words that are in the top 30 of every class = topic words, not sentiment words
TOP_N = 30
common_words = set.intersection(*[{w for w, _ in words_by_class[s].most_common(TOP_N)} for s in CLASS_NAMES])
print("Hidden from the clouds (common to all classes):", sorted(common_words))


def make_color_func(hue):
    # one hue per class, random darkness so words are readable on white
    def color_func(*args, **kwargs):
        return f"hsl({hue}, 65%, {random.randint(25, 45)}%)"
    return color_func


hues = {"Negative": 0, "Neutral": 215, "Positive": 130}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, s in zip(axes, CLASS_NAMES):
    freqs = {w: c for w, c in words_by_class[s].items() if w not in common_words}
    cloud = WordCloud(width=700, height=500, background_color="white", max_words=100,
                      color_func=make_color_func(hues[s]), random_state=RANDOM_STATE)
    cloud.generate_from_frequencies(freqs)
    ax.imshow(cloud, interpolation="bilinear")
    ax.set_title(f"{s} comments", fontsize=15, color=PALETTE[s])
    ax.axis("off")

plt.tight_layout()
plt.show()

**Reading the clouds**

* **Positive** is easy to recognise – words of praise and appreciation (*good, great, love, best, thank, nation*) stand out.
* **Negative** is led by complaint words (*poor, hate, fake, money, cant*).
* **Neutral** has no strong emotional words of its own. What stands out are reporting words (*says, said, minister, government, today, via*), i.e. people passing on news or asking something. Because it has no vocabulary of its own, I expect the model to struggle most around this class.

(The exact words can change a little depending on the NLTK version, but the overall picture should be the same.)

## 3 · Feature extraction with TF-IDF

Machine-learning models need numbers, not sentences. **TF-IDF (Term Frequency – Inverse Document Frequency)** turns each comment into a vector of numbers:

* **TF** – how often a word appears *in this comment*. More occurrences → more important for this comment.
* **IDF** – how *rare* the word is across *all* comments. Words that appear everywhere (like "get" or "time") get a low weight; rare, specific words (like "terrible" or "fantastic") get a high weight.
* **TF × IDF** – a word is important for a comment if it appears there often **and** is not common everywhere.

Compared with a plain word count (Bag-of-Words), TF-IDF stops very common words from dominating, which usually gives better results for text classification.

Settings I chose:

* `ngram_range=(1, 2)` – use single words **and** pairs of words. Pairs capture things like *"not good"* or *"love it"* that single words miss.
* `min_df=3` – ignore words that appear in fewer than 3 comments (typos, one-off words).
* `max_features=150000` – keep the vocabulary at a manageable size.
* `sublinear_tf=True` – use `1 + log(tf)` so a word repeated 10 times is not treated as 10× more important.

The vectoriser is fitted **only on the training data**, so no information from the test set leaks in.

In [ ]:
# 80/20 split. stratify keeps the same class proportions in train and test.
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"], df["label"],
    test_size=0.20, stratify=df["label"], random_state=RANDOM_STATE,
)
print(f"Train: {len(X_train_text):,} comments | Test: {len(X_test_text):,} comments")

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=150000, sublinear_tf=True)
X_train = tfidf.fit_transform(X_train_text)   # learn vocabulary + IDF from train only
X_test = tfidf.transform(X_test_text)         # test just uses what train learned

print("TF-IDF matrix (train):", X_train.shape)
print("Vocabulary size      :", len(tfidf.vocabulary_))

## 4 · Training the classifiers

I'm training three models so I can compare them properly:

1. **Multinomial Naive Bayes** – fast, simple probabilistic baseline that works well with word counts / TF-IDF.
2. **Logistic Regression** – a strong linear model; it also gives readable weights, so I can see which words push a prediction towards each class.
3. **Linear SVM (LinearSVC)** – finds the best separating boundary; usually very strong on high-dimensional sparse text data.

Because the classes are a bit imbalanced, Logistic Regression and SVM use `class_weight="balanced"` (Naive Bayes has no such option, it relies on class priors instead).

For each model I tried a few values of its main parameter (Naive Bayes `alpha`, `C` for the other two) and kept the best one. Strictly speaking, doing that with cross-validation on the training set would be cleaner than looking at the test split, so I list it under "ideas to improve" at the end.

Below is one helper function so all three models are evaluated in exactly the same way: accuracy, macro precision / recall / F1, the per-class report, and a confusion matrix (raw counts on the left, row-normalised on the right – the right one shows the recall of each class).

In [ ]:
results = []       # one dict per model, used later for the comparison table
fitted = {}        # model name -> (model object, predictions on test set)


def evaluate_model(name, model):
    # train
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    # predict + metrics
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, preds, average="macro", zero_division=0)
    _, _, f1_weighted, _ = precision_recall_fscore_support(y_test, preds, average="weighted", zero_division=0)

    results.append({
        "model": name, "accuracy": acc, "precision_macro": prec,
        "recall_macro": rec, "f1_macro": f1, "f1_weighted": f1_weighted,
        "train_seconds": round(train_time, 1),
    })
    fitted[name] = (model, preds)

    print(f"===== {name} =====")
    print(f"Accuracy: {acc:.4f} | Macro precision: {prec:.4f} | Macro recall: {rec:.4f} | Macro F1: {f1:.4f}")
    print(f"Training time: {train_time:.1f}s\n")
    print(classification_report(y_test, preds, target_names=CLASS_NAMES, digits=3))

    # confusion matrix (counts and row-normalised)
    cm = confusion_matrix(y_test, preds)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[0].set_title(f"{name} - confusion matrix (counts)")
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", cbar=False, ax=axes[1],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[1].set_title(f"{name} - confusion matrix (row-normalised)")
    for ax in axes:
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.show()

### Model 1 – Multinomial Naive Bayes

In [ ]:
# alpha is the smoothing value. I compared 0.05 to 1.0 - small values were much better than the default of 1.0.
evaluate_model("Naive Bayes", MultinomialNB(alpha=0.1))

### Model 2 – Logistic Regression

In [ ]:
# C controls regularisation (smaller = stronger). C=1 beat 3, 5 and 10 in my small comparison.
evaluate_model("Logistic Regression",
               LogisticRegression(C=1, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))

### Model 3 – Linear SVM

In [ ]:
# Same idea for the SVM: a small C (0.25) did better than 0.5 or 1.0.
evaluate_model("Linear SVM",
               LinearSVC(C=0.25, class_weight="balanced", random_state=RANDOM_STATE))

## 5 · Model comparison

I also add a very simple **rule-based baseline with TextBlob**. It uses a fixed dictionary of word polarities and needs no training. If my trained models cannot beat it, the training was not worth it. (The cell is skipped automatically if TextBlob is not installed.)

In [ ]:
ml_results = pd.DataFrame(results).set_index("model")

# --- rule-based baseline: TextBlob polarity -> class ---
baseline_row = None
try:
    from textblob import TextBlob

    def textblob_label(text):
        polarity = TextBlob(text).sentiment.polarity     # between -1 and +1
        if polarity < -0.05:
            return 0
        if polarity > 0.05:
            return 2
        return 1

    tb_preds = X_test_text.apply(textblob_label)
    p, r, f, _ = precision_recall_fscore_support(y_test, tb_preds, average="macro", zero_division=0)
    _, _, f_w, _ = precision_recall_fscore_support(y_test, tb_preds, average="weighted", zero_division=0)
    baseline_row = pd.DataFrame([{
        "accuracy": accuracy_score(y_test, tb_preds), "precision_macro": p, "recall_macro": r,
        "f1_macro": f, "f1_weighted": f_w, "train_seconds": 0.0,
    }], index=["TextBlob (rule-based)"])
except ImportError:
    print("TextBlob is not installed, skipping the baseline (pip install textblob).")

comparison = ml_results if baseline_row is None else pd.concat([ml_results, baseline_row])
comparison.round(4)

In [ ]:
metrics_to_plot = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
plot_df = comparison[metrics_to_plot].rename(columns={
    "accuracy": "Accuracy", "precision_macro": "Precision (macro)",
    "recall_macro": "Recall (macro)", "f1_macro": "F1 (macro)"})

ax = plot_df.plot(kind="bar", figsize=(10, 5), width=0.8, colormap="viridis", edgecolor="white")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Model comparison on the 20% test set")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=8, padding=2)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Which words drive the predictions?

Logistic Regression gives one weight per word for each class. The words with the highest weight for a class are the strongest evidence *for* that class – a nice way to check that the model learned something sensible and not just noise.

In [ ]:
lr_model = fitted["Logistic Regression"][0]
assert list(lr_model.classes_) == [0, 1, 2]     # so row i of coef_ matches CLASS_NAMES[i]

feature_names = np.array(tfidf.get_feature_names_out())

for i, name in enumerate(CLASS_NAMES):
    top_idx = np.argsort(lr_model.coef_[i])[-15:][::-1]
    print(f"{name:>8}: " + ", ".join(feature_names[top_idx]))

**What the weights show:** the positive and negative lists are full of clearly emotional words, so the model has learned sensible sentiment cues. The neutral list is usually less convincing – neutral is defined more by the *absence* of strong words than by words of its own, which fits what the word clouds showed.

## 6 · Error analysis

Numbers only tell half the story. Here I take the **best model** (by macro F1) and look at where it goes wrong. First the overall pattern of mistakes, then five real examples (one for each of the most common error types).

In [ ]:
best_name = ml_results["f1_macro"].idxmax()
best_preds = fitted[best_name][1]
print("Best model by macro F1:", best_name)

# put everything about the test set in one table
test_idx = X_test_text.index
err_df = pd.DataFrame({
    "original":   df.loc[test_idx, "text"].values,
    "clean":      X_test_text.values,
    "true":       y_test.map(label_map).values,
    "predicted":  pd.Series(best_preds).map(label_map).values,
})
errors = err_df[err_df["true"] != err_df["predicted"]].copy()

print(f"Misclassified: {len(errors):,} of {len(err_df):,} test comments ({len(errors) / len(err_df):.1%})")

pair_counts = errors.groupby(["true", "predicted"]).size().sort_values(ascending=False)
print("\nMost common mistakes (actual -> predicted):")
print(pair_counts.to_string())

neutral_share = ((errors["true"] == "Neutral") | (errors["predicted"] == "Neutral")).mean()
print(f"\n{neutral_share:.0%} of all mistakes involve the Neutral class (either as the true or the predicted label).")

In [ ]:
# Simple rule-based hints about why a comment might have been misclassified.
# This is only a guide - I read the examples myself in the discussion below.
negation_words = {"not", "no", "nor", "dont", "didnt", "doesnt", "cant", "wont", "isnt",
                  "wasnt", "couldnt", "wouldnt", "shouldnt", "never", "nothing"}


def possible_reasons(row):
    words = row["clean"].split()
    reasons = []
    if len(words) <= 4:
        reasons.append("very short text, hardly any sentiment words to work with")
    if len(words) >= 25:
        reasons.append("long comment, probably mixes positive and negative points")
    if "Neutral" in (row["true"], row["predicted"]):
        reasons.append("sits on the fuzzy border between neutral and an opinion")
    if {row["true"], row["predicted"]} == {"Positive", "Negative"}:
        reasons.append("opposite polarity - maybe sarcasm, mixed feelings or a questionable label")
    if negation_words & set(words):
        reasons.append("contains a negation, which a bag-of-words model only partly understands")
    return reasons or ["no obvious cue - ambiguous wording or noisy label"]


# one example from each of the 5 most common error types
examples = []
for (true_label, pred_label) in pair_counts.index[:5]:
    subset = errors[(errors["true"] == true_label) & (errors["predicted"] == pred_label)]
    examples.append(subset.sample(1, random_state=7).iloc[0])

for n, row in enumerate(examples, start=1):
    text = row["original"]
    if len(text) > 350:
        text = text[:350] + " ..."
    print(f"Example {n}")
    print(f"  Comment  : {text}")
    print(f"  Actual   : {row['true']}   |   Predicted: {row['predicted']}")
    print("  Possible reasons:")
    for reason in possible_reasons(row):
        print(f"    - {reason}")
    print()

### Why does the model get these wrong?

Reading through the examples (change `random_state=7` in the cell above to see other ones), the mistakes fall into a few groups that keep coming back:

1. **The neutral boundary is blurry.** Roughly seven in ten mistakes involve the neutral class (the exact share is printed above). That is natural, because neutral sits between the other two: a news-style sentence or a plain question can easily carry a slightly positive or negative tone, and the human label and the model's guess can both be reasonable.
2. **Very short comments.** With only two or three words there is simply not enough evidence, so the model falls back on whatever the few words usually mean in the training data.
3. **Sarcasm and mixed opinions.** "Great, another price hike" contains the word *great* but is negative. A bag-of-words model only sees words, not tone. Long comments that praise one thing and criticise another have the same problem.
4. **Negation and word order.** Bigrams help for patterns like "not good", but longer-distance negation ("I wouldn't say it was good") is still hard for TF-IDF features.
5. **Vocabulary the model doesn't know.** Not every comment is clean English. I found some written in another language entirely (Turkish, for example) and others in Hindi typed with English letters (e.g. *nhi*, *taklif*). Those words are rare or missing in the vocabulary, so they contribute almost nothing and the model is basically guessing.
6. **Label noise.** In a dataset of this size, some labels are arguable or simply wrong (the identical-text-different-label cases I removed earlier are proof that it happens). Part of the remaining error can never be fixed by a better model.

**Ideas to improve:** a transformer model (BERT / a multilingual or Hinglish-aware one) that understands context and word order, filtering or separately handling non-English comments, more careful handling of emojis and slang, proper hyper-parameter tuning with cross-validation (`GridSearchCV`) on the training data only, and collecting more neutral examples with clearer labelling guidelines.

## 7 · Conclusion

In [ ]:
ranked = ml_results.sort_values("f1_macro", ascending=False)
best = ranked.iloc[0]

print(f"Best model : {ranked.index[0]}")
print(f"Accuracy   : {best['accuracy']:.3f}")
print(f"Macro F1   : {best['f1_macro']:.3f}\n")

print("Ranking by macro F1:")
for rank, (name, row) in enumerate(ranked.iterrows(), start=1):
    print(f"  {rank}. {name:<20} F1 = {row['f1_macro']:.3f} | accuracy = {row['accuracy']:.3f} | trained in {row['train_seconds']}s")

if baseline_row is not None:
    tb = baseline_row.iloc[0]
    print(f"\nTextBlob baseline: F1 = {tb['f1_macro']:.3f} | accuracy = {tb['accuracy']:.3f}")
    print(f"The best trained model beats it by {best['f1_macro'] - tb['f1_macro']:+.3f} macro F1.")

# per-class recall of the best model: which sentiment is easiest / hardest?
_, _, rec_per_class, _ = precision_recall_fscore_support(y_test, fitted[ranked.index[0]][1], zero_division=0)
print("\nRecall of the best model by class:")
for name, r in zip(CLASS_NAMES, rec_per_class):
    print(f"  {name:<9}: {r:.3f}")

### Summary of findings

* **Linear models on TF-IDF (Logistic Regression and Linear SVM) clearly beat Naive Bayes**, and they score very close to each other. Naive Bayes is still a good, very fast baseline, but it is the weakest here: it assumes every word is independent, and it leans towards the biggest class (positive), so it misses many negative comments. The ranking printed above shows the exact numbers – I pick the model at the top (highest macro F1) as the final one. Since the top two are so close, the Linear SVM's much shorter training time is a practical bonus.
* The TextBlob line printed above (if TextBlob is installed) shows how far a general-purpose dictionary gets without any training. The gap to the trained models is the value of learning from labelled data from *this* domain, which includes slang, sarcasm and topic words that a fixed word list does not know.
* The per-class recall printed above shows which sentiment the final model finds easiest and hardest, and the error analysis showed that the neutral class is involved in most of the mistakes.
* I trust **macro F1** more than accuracy here because the classes are unequal in size.
* Duplicate comments were removed before splitting, so these scores are an honest estimate and not inflated by leakage.

### Real-world applications

* **Brand and product monitoring** – track how people feel about a product, a feature (for example a payment app) or a launch, in real time, and alert the team when negative comments suddenly rise.
* **Public-opinion tracking** – summarise reactions to a policy, speech or election event from social media, instead of reading thousands of posts by hand.
* **Customer-support triage** – route very negative messages to a human agent first.
* **Creator / community moderation** – help YouTube or forum owners find the constructive and the toxic comments quickly.

### Limitations

The model reads words, not meaning: it struggles with sarcasm, mixed opinions, and Hinglish. It also learned some topic-specific vocabulary from this dataset, so it should be re-checked (or re-trained) before using it on a very different domain such as hotel reviews or medical text. For production use, I would try a fine-tuned transformer and compare it against these baselines.